
# 🔎 Core Vision Perfect V1 — 07 · OCR PaddleOCR — CA 3/3 (bản c)

Nâng cấp đôi mắt đọc chữ: đọc lại TOÀN BỘ keyframe bằng **PaddleOCR (PP-OCRv5,
tiếng Việt)** — thay kho EasyOCR cũ vốn nhiễu (bộ tinh chỉnh trọng số từng phải
dìm kênh OCR xuống gần 0). Chữ nung trên khung hình (banner, tiêu đề, chyron)
là tín hiệu KIS đắt giá nhất khi đọc CHUẨN dấu tiếng Việt. Nhanh hơn captions
nhiều lần (~2–5 giờ/ca GPU); cùng bộ giáp an toàn round-52..58 như 05/06.


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  1 · PARAMS — the ONLY cell you may need to edit                 ║
# ╚══════════════════════════════════════════════════════════════════╝
DRIVE_PROJECT_DIR = "AIC2025"        # MyDrive/<this>/{data, artifacts}
REPO_URL  = "https://github.com/ledinhminhquan/Core-Vision_Perfect_V1.git"
REPO_REF  = "main"

# Which dense encoders to build indexes for (order = ensemble order).
#   "siglip2"          multilingual default (needed for training too)
#   "openclip"         English lane (DFN5B ViT-H/14-378) — strongest with translation
#   "qwen_embed"       optional HEAVY lane (Qwen embedding tower — strong, slow)
#   "provided_clip32"  organiser features — instant, no GPU (L-batches only);
#                      auto-added in the catalog cell when clip-features-32 exists
EMBED_MODELS = ["siglip2", "openclip"]

# Copy keyframes from Drive → local disk before embedding (much faster I/O).
COPY_KEYFRAMES_LOCAL = True

# Aux indexes to build (each is resumable; captions are the slowest).
RUN_OCR, RUN_ASR, RUN_CAPTIONS = True, True, True
CAPTION_STRIDE = 4                   # caption mỗi keyframe thứ 4 (round-16: đủ dày
#   cho kênh recall BM25 mà nhanh gấp đôi stride 2. NÂNG stride luôn an toàn với
#   resume: video đã caption ở stride nhỏ hơn vẫn được tính là XONG ở stride lớn
#   hơn. Mọi phiên chạy song song PHẢI dùng CÙNG một stride.

# K-batch shot detection: install TransNetV2 (the winning-team detector) for
# keyframe self-extraction. Installed --no-deps (Colab torch is never touched);
# without it extraction falls back to PySceneDetect automatically. (nb01 only)
INSTALL_TRANSNETV2 = True

# Force-rebuild toggles — mặc định False = resume/skip khi artifact đã có.
FORCE_CATALOG    = False             # rebuild the catalog parquet
FORCE_EMBED      = False             # re-embed every keyframe
FORCE_INDEX      = False             # rebuild the FAISS indexes
FORCE_AUX        = False             # redo OCR/ASR/captions from scratch
FORCE_TEXT_INDEX = False             # rebuild the persisted BM25 text index

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("params ok")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  2 · Mount Drive + folder layout + preflight write test          ║
# ╚══════════════════════════════════════════════════════════════════╝
import os, shutil, subprocess, time
from pathlib import Path

from google.colab import drive

_MP = "/content/drive"

def _drive_alive() -> bool:
    try:
        return Path(_MP, "MyDrive").exists()
    except OSError:
        return False

def _ensure_drive():
    """Mount / HỒI SINH Drive FUSE — dùng ở mọi cell dài hơi phía sau.

    Round-19 (live run 8): daemon DriveFS chết để lại mountpoint 'bẩn' →
    drive.mount kêu 'Mountpoint must not already contain files' và cả
    force_remount cũng bó tay. Trình tự cứu đúng: (1) fusermount -uz gỡ
    mount chết; (2) CHỈ khi chắc chắn không còn mount (os.path.ismount ==
    False — lúc này các entry trong mountpoint là RÁC LOCAL trên đĩa VM,
    không phải Drive thật) mới dọn sạch chúng; (3) mount lại.
    """
    for _try in range(4):
        if _drive_alive():
            return
        if _try:
            print(f"⚠ Drive FUSE chưa sống — hồi sinh (lần {_try}/3) ...")
        try:
            if os.path.ismount(_MP):
                subprocess.run(["fusermount", "-uz", _MP], capture_output=True)
                time.sleep(2)
            if os.path.isdir(_MP) and not os.path.ismount(_MP):
                for _c in os.listdir(_MP):     # rác local — KHÔNG phải Drive
                    _p = os.path.join(_MP, _c)
                    shutil.rmtree(_p, ignore_errors=True) if os.path.isdir(_p) \
                        else os.unlink(_p)
            drive.mount(_MP, force_remount=bool(_try))
        except Exception as _e:  # noqa: BLE001 — thử tiếp vòng sau
            print("   mount lỗi:", _e)
            time.sleep(5)
    if not _drive_alive():
        raise RuntimeError(
            "Không mount được Google Drive sau 4 lần thử — Runtime ▸ "
            "Disconnect and delete runtime rồi Run all lại (tiến độ đã lưu "
            "trên Drive còn nguyên).")

_ensure_drive()
assert Path("/content/drive/MyDrive").exists(), "Drive mount failed — rerun this cell"

PROJECT   = Path("/content/drive/MyDrive") / DRIVE_PROJECT_DIR
DATA_DIR  = PROJECT / "data"           # organiser dataset (merged packages)
ARTIFACTS = PROJECT / "artifacts"      # everything we build → survives disconnects
for p in (DATA_DIR, ARTIFACTS):
    p.mkdir(parents=True, exist_ok=True)

# PREFLIGHT (v12): Drive PHẢI ghi/đọc được — quota đầy hay mất quyền thì
# dừng NGAY tại đây thay vì hỏng giữa chừng sau 2 giờ chạy.
_probe = ARTIFACTS / f"_write_test_{int(time.time())}.tmp"
try:
    _probe.write_text("ok", encoding="utf-8")
    assert _probe.read_text(encoding="utf-8") == "ok"
    _probe.unlink()
    print("✅ Drive write test: OK")
except Exception as e:
    raise RuntimeError(
        f"❌ Không ghi được vào Drive ({ARTIFACTS}): {e!r}\n"
        "Kiểm tra dung lượng (quota) Google Drive và quyền truy cập thư mục, "
        "rồi chạy lại ô này."
    ) from e

# DATA-PRESENCE GATE (round-20, live run 9): trên VM mới, DriveFS có thể liệt
# kê data/ ra RỖNG suốt vài phút đầu (metadata sync lười) — mkdir exist_ok ở
# trên còn CHE mất triệu chứng, để cell 7 chết khó hiểu với "Keyframes folder
# not found". Poll tới 3 phút (mỗi listdir là một cú hích ép DriveFS fetch);
# hết kiên nhẫn thì dừng TO với chẩn đoán rõ ràng.
_t0 = time.time()
_data_ok = False
while time.time() - _t0 < 180:
    try:
        if any(DATA_DIR.iterdir()):
            _data_ok = True
            break
    except OSError:
        pass
    print(f"⏳ data/ đang rỗng — đợi DriveFS sync metadata ({int(time.time() - _t0)}s) ...")
    time.sleep(10)
if not _data_ok:
    raise RuntimeError(
        "data/ trên Drive vẫn RỖNG sau 3 phút chờ. Ba nguyên nhân thường gặp:\n"
        "  1) Phiên Colab đăng nhập NHẦM tài khoản Google (kiểm tra avatar góc "
        f"phải trên) — phải là tài khoản có MyDrive/{DRIVE_PROJECT_DIR}/data;\n"
        "  2) DriveFS sync quá chậm — Runtime ▸ Disconnect and delete runtime "
        "rồi Run all lại trên máy mới;\n"
        "  3) Lần chạy đầu tiên mà chưa upload dữ liệu — ném các zip của BTC "
        f"vào MyDrive/{DRIVE_PROJECT_DIR}/data trước (docs/DRIVE_SETUP.md).\n"
        "KHÔNG có gì bị mất — dữ liệu vẫn nằm nguyên trên Drive của tài khoản đúng.")
print(f"✅ data/ nhìn thấy dữ liệu sau {int(time.time() - _t0)}s")

# HF + pip caches on Drive → models/wheels download once, not per session.
os.environ["HF_HOME"] = str(ARTIFACTS / "hf_cache")
os.environ["PIP_CACHE_DIR"] = str(ARTIFACTS / "pip_cache")
for _d in (os.environ["HF_HOME"], os.environ["PIP_CACHE_DIR"]):
    Path(_d).mkdir(parents=True, exist_ok=True)

import shutil
free_gb = shutil.disk_usage(str(PROJECT)).free / 1e9
print(f"Project: {PROJECT}")
print(f"Drive free space: {free_gb:.0f} GB")
if free_gb < 20:
    print("⚠ Less than 20 GB free on Drive — embeddings/checkpoints may not fit!")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  3 · Get repo + install dependencies (v12 discipline)            ║
# ╚══════════════════════════════════════════════════════════════════╝
# Quy tắc (học từ notebook Toxicity v12):
#   * check version qua importlib.metadata — KHÔNG import package trước khi
#     nâng cấp (import sớm sẽ ghim version cũ vào sys.modules);
#   * KHÔNG BAO GIỜ đụng torch/torchvision/torchaudio của Colab;
#   * chỉ cài đúng những gói thiếu/sai version (--prefer-binary);
#   * sau khi cài: `pip check` + micro-fix (tối đa 2 vòng, không crash),
#     rồi purge sys.modules TRƯỚC khi import cvp.
FORCE_REINSTALL_DEPS = False

import re, subprocess, sys
from pathlib import Path

REPO_DIR = Path("/content/Core-Vision_Perfect_V1")

def _run(cmd, show=None, **kw):
    # `show` masks credentials in the echoed command — a PAT-carrying clone
    # URL must NEVER be printed into the saved notebook output.
    print("$", " ".join(map(str, show or cmd)))
    return subprocess.run([str(c) for c in cmd], check=False, **kw).returncode

def _pip(args):
    return _run([sys.executable, "-m", "pip", *args])

# Private repo? Add a fine-grained PAT as Colab secret "GITHUB_TOKEN"
# (Contents: Read-only on this repo) and ENABLE its notebook-access toggle.
clone_url, _tok = REPO_URL, None
try:
    from google.colab import userdata
    _tok = userdata.get("GITHUB_TOKEN")
except Exception as _e:
    print(f"⚠ KHÔNG đọc được secret GITHUB_TOKEN ({type(_e).__name__}) — repo "
          "private sẽ KHÔNG clone được. Kiểm tra: 🔑 panel có secret tên đúng "
          "y hệt GITHUB_TOKEN và công tắc 'Notebook access' đã BẬT chưa?")
if _tok and clone_url.startswith("https://github.com/"):
    clone_url = clone_url.replace("https://", f"https://{_tok}@")
    print(f"GITHUB_TOKEN: loaded ({len(_tok)} chars, {_tok[:11]}…)")
elif not _tok:
    print("⚠ GITHUB_TOKEN trống/vắng mặt — thử clone KHÔNG xác thực "
          "(chắc chắn fail nếu repo private).")

if REPO_DIR.exists():
    _run(["git", "-C", REPO_DIR, "fetch", "--all", "-q"])
    _run(["git", "-C", REPO_DIR, "checkout", REPO_REF, "-q"])
    _run(["git", "-C", REPO_DIR, "pull", "-q"])
else:
    rc = _run(["git", "clone", "--branch", REPO_REF, clone_url, REPO_DIR],
              show=["git", "clone", "--branch", REPO_REF, REPO_URL, REPO_DIR])
    if rc != 0:  # private repo / no network → fall back to a Drive copy
        print("⚠ Clone THẤT BẠI. Nguyên nhân thường gặp, theo thứ tự:\n"
              "  1) Secret GITHUB_TOKEN sai tên / chưa bật Notebook access "
              "(xem cảnh báo phía trên);\n"
              "  2) PAT sai/hết hạn/thiếu quyền — cần fine-grained PAT với "
              "Contents: Read-only cấp cho ĐÚNG repo này;\n"
              "  3) Mạng Colab trục trặc tạm thời — chạy lại cell.")
        drive_copy = Path("/content/drive/MyDrive") / DRIVE_PROJECT_DIR / "Core-Vision_Perfect_V1"
        assert drive_copy.exists(), (
            "Clone failed and no Drive copy found. Either make the GitHub repo "
            f"reachable or upload the repo folder to {drive_copy}"
        )
        import shutil as _sh
        _sh.copytree(drive_copy, REPO_DIR)
        print("Using repo copy from Drive")

try:
    from packaging.requirements import Requirement
except ImportError:
    _pip(["install", "-q", "packaging"])
    from packaging.requirements import Requirement
from importlib.metadata import PackageNotFoundError
from importlib.metadata import version as _meta_version

# Parse requirements-colab.txt; strip any torch* line (Colab rule #1: the
# preinstalled torch/torchvision/torchaudio build must never be touched).
reqs = []
for _line in (REPO_DIR / "requirements-colab.txt").read_text(encoding="utf-8").splitlines():
    _line = _line.split("#", 1)[0].strip()
    if not _line:
        continue
    try:
        _r = Requirement(_line)
    except Exception:
        print("⚠ bỏ qua requirement không parse được:", _line)
        continue
    if _r.name.lower().replace("-", "_").startswith("torch"):
        print("skip (never touch Colab torch):", _line)
        continue
    reqs.append(_r)

def _satisfied(r):
    """Installed + in range — via importlib.metadata, WITHOUT importing it."""
    try:
        v = _meta_version(r.name)
    except PackageNotFoundError:
        return False
    return (not r.specifier) or r.specifier.contains(v, prereleases=True)

missing = [r for r in reqs if FORCE_REINSTALL_DEPS or not _satisfied(r)]
did_install = bool(missing)
if missing:
    print(f"installing {len(missing)} package(s):", ", ".join(r.name for r in missing))
    _pip(["install", "-q", "--prefer-binary", *[str(r) for r in missing]])
else:
    print("dependencies satisfied — no pip install needed")

_pip(["install", "-q", "-e", str(REPO_DIR), "--no-deps"])

# faiss: gpu wheel with cpu fallback (metadata check — no import)
def _installed(*names):
    for n in names:
        try:
            _meta_version(n)
            return n
        except PackageNotFoundError:
            pass
    return None

if _installed("faiss-gpu-cu12", "faiss-gpu", "faiss-cpu", "faiss") is None:
    if _pip(["install", "-q", "faiss-gpu-cu12"]) != 0:
        _pip(["install", "-q", "faiss-cpu"])
    did_install = True

# `pip check` + micro-fixes for known conflicts (max 2 rounds, then warn)
def _pip_check():
    r = subprocess.run([sys.executable, "-m", "pip", "check"],
                       capture_output=True, text=True)
    return r.returncode, ((r.stdout or "") + "\n" + (r.stderr or "")).strip()

if did_install:
    rc, out = _pip_check()
    for _round in (1, 2):
        if rc == 0:
            break
        # pip's two REAL formats (round-3 fix L-R3-8 — the old regex missed the
        # version-conflict wording so that repair branch never ran):
        #   "pkgA 1.0 requires pkgB, which is not installed."
        #   "pkgA 1.0 has requirement pkgB<2,>=1, but you have pkgB 3.0."
        _specs = sorted({
            m.strip()
            for m in re.findall(
                r"(?:requires|has requirement) (.+?), (?:but you have|which is not installed)", out)
            if not m.strip().lower().startswith("torch")
        })
        if not _specs:
            break
        print(f"pip check micro-fix (round {_round}):", ", ".join(_specs))
        _pip(["install", "-q", "--prefer-binary", *_specs])
        rc, out = _pip_check()
    print("pip check: OK" if rc == 0 else f"⚠ pip check còn cảnh báo (không chặn):\n{out}")

# Purge stale sys.modules of upgraded packages BEFORE importing cvp (v12).
# ONLY the packages actually (re)installed THIS run (round-11): purging every
# requirement dropped numpy/pandas from sys.modules while torch still held
# references to the old modules — the "NumPy module was reloaded" warning.
if did_install:
    _ALIAS = {"pillow": "pil", "pyyaml": "yaml", "opencv_python_headless": "cv2",
              "open_clip_torch": "open_clip", "scikit_learn": "sklearn"}
    _roots = {r.name.lower().replace("-", "_") for r in missing} | {"cvp", "faiss"}
    _roots |= {_ALIAS[n] for n in _roots & set(_ALIAS)}
    _purged = [m for m in list(sys.modules)
               if m.split(".", 1)[0].lower().replace("-", "_") in _roots]
    for _m in _purged:
        sys.modules.pop(_m, None)
    if _purged:
        print(f"purged {len(_purged)} stale sys.modules entries")

    # Sanity (round-11): the HF stack must import cleanly in a FRESH
    # interpreter — a broken hub/accelerate pairing must surface HERE with an
    # actionable message, not 5 cells later as a cryptic circular import.
    _rc = _run([sys.executable, "-c", "import transformers, accelerate"])
    if _rc != 0:
        print("⚠ transformers/accelerate KHÔNG import được — thường do phiên cài "
              "này đã hạ cấp huggingface-hub dưới mức accelerate cần. Cách sửa "
              "sạch nhất: Runtime ▸ Disconnect and delete runtime, rồi Run all "
              "lại từ đầu (mọi tiến độ đã nằm trên Drive, không mất gì).")

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
import cvp
print("cvp", cvp.__version__, "ready")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  4 · Point cvp at the data + GPU setup (TF32 / SDPA / bf16)      ║
# ╚══════════════════════════════════════════════════════════════════╝
import os, torch

os.environ["CVP_PATHS__DATA_ROOT"]      = str(DATA_DIR)
os.environ["CVP_PATHS__ARTIFACTS_ROOT"] = str(ARTIFACTS)
os.environ["CVP_SETTINGS"] = str(REPO_DIR / "configs" / "settings.yaml")

print("torch", torch.__version__, "| CUDA build", torch.version.cuda)
print("GPU available:", torch.cuda.is_available())
GPU_NAME, VRAM_GB, USE_BF16 = "cpu", 0.0, False
if torch.cuda.is_available():
    GPU_NAME = torch.cuda.get_device_name(0)
    VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
    USE_BF16 = torch.cuda.is_bf16_supported()
    # TF32 fast paths (new API with old fallback)
    try:
        torch.backends.cuda.matmul.fp32_precision = "tf32"
        torch.backends.cudnn.conv.fp32_precision = "tf32"
    except Exception:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
    for fn in ("enable_flash_sdp", "enable_mem_efficient_sdp"):
        if hasattr(torch.backends.cuda, fn):
            getattr(torch.backends.cuda, fn)(True)
print(f"GPU: {GPU_NAME} | VRAM {VRAM_GB:.0f} GB | bf16={USE_BF16}")

# Colab secrets → env (optional: Gemini query enhancement/VQA, HF pushes).
# GOOGLE_API_KEY is the name Colab's built-in "Gemini API key ▸ Import from
# Google AI Studio" button creates — the engine accepts either spelling.
try:
    from google.colab import userdata
    for _sec in ("GEMINI_API_KEY", "GOOGLE_API_KEY", "HF_TOKEN"):
        try:
            _v = userdata.get(_sec)
            if _v:
                os.environ[_sec] = _v
                print(f"secret {_sec}: loaded")
        except Exception:
            pass
except ImportError:
    pass

from cvp.config import load_settings
from cvp.utils.logging import setup_logging
settings = load_settings()
setup_logging("INFO")
print("data_root      =", settings.paths.data_root)
print("artifacts_root =", settings.paths.artifacts_root)

In [ ]:
# ── 6 · Materialize data → local disk TỪ ZIP GỐC (nhanh + miễn nhiễm FUSE) ──
# Round-14 (live-run 5): copytree 177k JPG lẻ qua Drive FUSE mất 3h+ rồi làm
# SẬP luôn cả mount ([Errno 107] Transport endpoint is not connected — mọi
# file sau đó đọc ra ENOENT). Chiến lược mới: copy CÁC FILE ZIP về local
# (ít file, to, đọc tuần tự — đúng kiểu I/O FUSE làm tốt) rồi giải nén tại
# chỗ — nhanh hơn nhiều lần, resume theo TỪNG zip, tự remount khi FUSE chết.
# Nội dung chỉ-có-trên-Drive (keyframes K-batch tự cắt, csv tái dựng…) được
# merge bù ở pha 2. Embedding/OCR/caption đọc 177k JPG từ local như cũ.
import re as _re
import shutil, time, zipfile
from pathlib import Path

# _ensure_drive/_drive_alive: bản HARDENED định nghĩa ở Ô 2 (round-19) —
# biết gỡ mount chết (fusermount -uz) + dọn mountpoint bẩn trước khi mount lại.
_ensure_drive()          # verify-R14: gate dưới stat qua FUSE — mount phải sống

def _kf_visible() -> bool:
    """DriveFS trên VM mới có thể thấy data/ nhưng CHƯA thấy subdir keyframes/
    (round-28, live nb03 run 5: gate này rơi nhầm sang nhánh Drive-direct rồi
    chết ở catalog). listdir cha = cú hích ép nạp metadata; zip Keyframes*
    cũng được chấp nhận — materialize vốn bung từ zip, không cần dir Drive."""
    try:
        list(DATA_DIR.iterdir())
        if (DATA_DIR / "keyframes").exists():
            return True
        _zn = [p.name.lower().replace("_", "-").replace(" ", "-")
               for p in DATA_DIR.glob("*.zip")]
        return any(n.startswith(("keyframes", "keyframe", "key-frames")) for n in _zn)
    except OSError:
        return False

_kf_ok = False
if COPY_KEYFRAMES_LOCAL:
    for _w in range(12):                       # tới 2 phút
        if _kf_visible():
            _kf_ok = True
            break
        print(f"⏳ DriveFS chưa thấy keyframes/ hay Keyframes*.zip — đợi ({_w * 10}s) ...")
        time.sleep(10)
    if not _kf_ok:
        print("⚠ 2 phút vẫn không thấy keyframes/ lẫn zip nguồn — rơi về đọc "
              "thẳng Drive (CHẬM; nếu bất thường: Disconnect and delete runtime).")
if COPY_KEYFRAMES_LOCAL and _kf_ok:
    LOCAL_DATA = Path("/content/data")
    LOCAL_DATA.mkdir(exist_ok=True)
    _ZCACHE = Path("/content/__zip_cache")
    _ZCACHE.mkdir(exist_ok=True)
    _VID_DIR_RE = _re.compile(r"^[A-Z]\d{2}_V\d{3}$")

    def _zip_family(zname: str):
        # CÙNG thứ tự ưu tiên với guess_dest ở ô 5 — một zip phải về đúng
        # MỘT family ở cả hai ô. Videos* trả None: video ở lại Drive (symlink).
        z = zname.lower().replace("_", "-").replace(" ", "-")
        if "map" in z and "keyframe" in z:                        return "map-keyframes"
        if "clip-feature" in z or "features-32" in z:             return "clip-features-32"
        if "media-info" in z or "metadata" in z:                  return "media-info"
        if "object" in z:                                         return "objects"
        if z.startswith(("keyframes", "keyframe", "key-frames")): return "keyframes"
        return None

    def _walk_wrapper(root: Path) -> Path:
        # bỏ các folder bọc ngoài thật sự (Keyframes_L26/keyframes/…) nhưng
        # không bao giờ nhầm một payload dir dạng L21_V001 đơn độc là wrapper
        src = root
        while True:
            ch = list(src.iterdir())
            if len(ch) == 1 and ch[0].is_dir() and not _VID_DIR_RE.match(ch[0].name):
                src = ch[0]
                continue
            return src

    def _merge_into(src: Path, dest: Path) -> int:
        """Move src/* vào dest — không ghi đè, đi sâu 1 cấp cho dir trùng."""
        kept = 0
        dest.mkdir(parents=True, exist_ok=True)
        for item in src.iterdir():
            target = dest / item.name
            if not target.exists():
                shutil.move(str(item), str(target))
            elif item.is_dir() and target.is_dir():
                for sub in item.iterdir():
                    st = target / sub.name
                    if not st.exists():
                        shutil.move(str(sub), str(st))
                    else:
                        kept += 1
            else:
                kept += 1
        return kept

    for _sub in ("keyframes", "map-keyframes", "media-info", "objects", "clip-features-32"):
        dst = LOCAL_DATA / _sub
        _stamp = LOCAL_DATA / f".materialized-{_sub}"
        if _stamp.exists():
            # verify-R14: stamp KHÔNG được che zip mới upload giữa session —
            # còn zip matching chưa có marker local thì phải bung bổ sung.
            _ensure_drive()
            _new = [z for z in sorted(DATA_DIR.glob("*.zip"))
                    if _zip_family(z.name) == _sub
                    and not (LOCAL_DATA / f".unzipped-{_sub}-{z.stem}").exists()]
            if not _new:
                print(f"{_sub}: đã materialize trong session này — skip")
                continue
            print(f"{_sub}: {len(_new)} zip mới sau lần materialize trước → bung bổ sung")
        if dst.exists():
            for stale in dst.glob("*.__tmp"):
                shutil.rmtree(stale, ignore_errors=True) if stale.is_dir() else stale.unlink()

        # PHA 1 — bung từ zip nguồn (marker LOCAL theo từng zip → resume mịn;
        # crash giữa merge không sao: lần sau bung lại, merge chỉ bù file thiếu)
        _ensure_drive()
        for zp in sorted(DATA_DIR.glob("*.zip")):
            if _zip_family(zp.name) != _sub:
                continue
            _done = LOCAL_DATA / f".unzipped-{_sub}-{zp.stem}"
            if _done.exists():
                continue
            t0 = time.time()
            lz = _ZCACHE / zp.name
            tmp_root = _ZCACHE / "__tmp_extract"
            for _attempt in (1, 2, 3):
                try:
                    # verify-R14: MỌI syscall chạm FUSE (stat, copyfile) phải
                    # nằm TRONG retry — zip trước mất nhiều phút extract thuần
                    # local, FUSE có thể chết trong cửa sổ đó.
                    _ensure_drive()
                    _free = shutil.disk_usage("/content").free
                    if _free < zp.stat().st_size * 2.2 + 5e9:
                        raise RuntimeError(          # không retry lỗi hết disk
                            f"Disk local sắp đầy ({_free / 1e9:.0f} GB) — không đủ "
                            f"chỗ bung {zp.name}. Runtime ▸ Disconnect and delete "
                            "runtime để lấy máy mới, hoặc đặt "
                            "COPY_KEYFRAMES_LOCAL=False (chậm hơn nhiều).")
                    shutil.copyfile(zp, lz)             # 1 file to, đọc tuần tự
                    if tmp_root.exists():
                        shutil.rmtree(tmp_root)
                    with zipfile.ZipFile(lz) as z:      # CRC check từng member
                        z.extractall(tmp_root)
                    break
                except (OSError, zipfile.BadZipFile) as e:
                    print(f"   ⚠ {zp.name}: {e!r} — thử lại ({_attempt}/3)")
                    if _attempt == 3:
                        raise
                    time.sleep(5)
            kept = _merge_into(_walk_wrapper(tmp_root), dst)
            shutil.rmtree(tmp_root, ignore_errors=True)
            lz.unlink(missing_ok=True)                  # trả disk ngay
            _done.touch()
            print(f"   {zp.name} → local {_sub}/ ({time.time() - t0:.0f}s"
                  + (f", giữ {kept} mục trùng)" if kept else ")"))

        # PHA 2 — merge phần CHỈ có trên Drive (K-batch tự cắt, upload tay…):
        # 1 lần listdir + exists-check local là rẻ; copy lẻ chỉ cho phần thiếu.
        added = 0
        srcD = DATA_DIR / _sub
        # verify-R14: family chỉ-có-folder (không zip nguồn) → pha 1 chưa hề
        # tạo dst; copy2 vào parent chưa tồn tại sẽ FileNotFoundError.
        dst.mkdir(parents=True, exist_ok=True)
        for _attempt in (1, 2, 3):
            try:
                _ensure_drive()                 # srcD.exists cũng chạm FUSE
                if srcD.exists():
                    for item in sorted(srcD.iterdir()):
                        if item.name.startswith(".unzipped-") or item.name.endswith(".__tmp"):
                            continue
                        target = dst / item.name
                        if target.exists():
                            continue
                        tmp_target = dst / (item.name + ".__tmp")
                        if tmp_target.is_dir():
                            shutil.rmtree(tmp_target)
                        elif tmp_target.exists():
                            tmp_target.unlink()
                        (shutil.copytree if item.is_dir() else shutil.copy2)(item, tmp_target)
                        tmp_target.rename(target)
                        added += 1
                break
            except OSError as e:
                print(f"   ⚠ merge Drive-extras {_sub}: {e!r} — thử lại ({_attempt}/3)")
                if _attempt == 3:
                    raise
                time.sleep(5)
        _stamp.touch()
        _n = sum(1 for _ in dst.iterdir())
        print(f"{_sub}: sẵn sàng local ({_n} mục"
              + (f", +{added} bù từ Drive" if added else "") + ")")
    # INTEGRITY + SELF-HEAL (round-11/12, live-run lessons): Google Drive FUSE
    # can serve freshly-written files back EMPTY (buffered writes lost when a
    # session dies mid-sync). Round-11 hit 873 header-less map csvs; round-12
    # hit empty clip-features .npy files that killed the provided_clip32 lane
    # AFTER 8h of GPU work. Validate every LOCAL small-file artifact and heal
    # broken ones straight FROM THE SOURCE ZIP (uploaded long ago = reliably
    # synced), repairing the Drive copy too.
    import zipfile as _zf
    import numpy as _np

    def _bad_csv(f):
        try:
            if f.stat().st_size < 40:
                return True
            with open(f, encoding="utf-8-sig") as fh:
                return sum(1 for _ in fh) < 2          # header only / empty
        except OSError:
            return True

    def _bad_npy(f):
        try:
            if f.stat().st_size < 90:                  # npy header alone is ~64B
                return True
            return _np.load(f, mmap_mode="r").shape[0] == 0
        except Exception:
            return True

    def _bad_empty(f):
        try:
            return f.stat().st_size == 0
        except OSError:
            return True

    # (subdir, glob, zip-name matcher, validator, key depth 1=basename 2=vid/name)
    _HEAL_SPECS = [
        ("map-keyframes", "*.csv",
         lambda z: "map" in z and "keyframe" in z, _bad_csv, 1),
        ("clip-features-32", "*.npy",
         lambda z: "clip-feature" in z or "features-32" in z, _bad_npy, 1),
        ("media-info", "*.json",
         lambda z: "media-info" in z or "metadata" in z, _bad_empty, 1),
        ("objects", "*/*.json",
         lambda z: "object" in z, _bad_empty, 2),
    ]
    for _sub, _pat, _match, _isbad, _depth in _HEAL_SPECS:
        _dirL = LOCAL_DATA / _sub
        if not _dirL.is_dir():
            continue
        _key = (lambda p: p.name) if _depth == 1 else (lambda p: f"{p.parent.name}/{p.name}")
        _bad = [f for f in sorted(_dirL.glob(_pat)) if _isbad(f)]
        if not _bad:
            print(f"{_sub} integrity: OK")
            continue
        print(f"⚠ {len(_bad)} file LOCAL rỗng/hỏng trong {_sub}/ (Drive FUSE mất "
              "dữ liệu?) — tự phục hồi từ zip gốc ...")
        # Zip handles opened ONCE per family (round-13): re-opening a Drive
        # zip per bad file would stall for hours on a family-scale corruption.
        _ensure_drive()                        # verify-R14: ZipFile đọc qua FUSE
        _members, _open_zips = {}, []
        for _z in DATA_DIR.glob("*.zip"):
            _zl = _z.name.lower().replace("_", "-")
            if _match(_zl):
                _zh = _zf.ZipFile(_z)
                _open_zips.append(_zh)
                for _n in _zh.namelist():
                    if not _n.endswith("/"):
                        _parts = Path(_n).parts
                        _members["/".join(_parts[-_depth:])] = (_zh, _n)
        _healed = 0
        for f in _bad:
            _srcz = _members.get(_key(f))
            if not _srcz:
                continue
            _data = _srcz[0].read(_srcz[1])
            if not _data:
                continue
            f.write_bytes(_data)                       # heal LOCAL
            _drv = DATA_DIR / _sub / _key(f)           # heal DRIVE too
            try:
                if not _drv.exists() or _isbad(_drv):
                    _tmpf = _drv.parent / (_drv.name + ".__tmp")
                    _tmpf.write_bytes(_data)
                    _tmpf.replace(_drv)
            except OSError:
                pass
            _healed += 1
        for _zh in _open_zips:
            _zh.close()
        print(f"   phục hồi {_healed}/{len(_bad)}")
        _still = [_key(f) for f in _bad if _isbad(f)]
        if _still:
            raise RuntimeError(
                f"{len(_still)} file trong {_sub}/ vẫn hỏng sau phục hồi "
                f"(vd {_still[:3]}) — kiểm tra zip nguồn còn trong data/ trên "
                "Drive (đừng xóa zip!) rồi chạy lại ô này.")
    # videos stay on Drive (huge); link them in
    _ensure_drive()
    if (DATA_DIR / "videos").exists() and not (LOCAL_DATA / "videos").exists():
        (LOCAL_DATA / "videos").symlink_to(DATA_DIR / "videos")
    import os
    os.environ["CVP_PATHS__DATA_ROOT"] = str(LOCAL_DATA)
    from cvp.config import load_settings
    settings = load_settings()
    print("data_root now:", settings.paths.data_root)
else:
    print("using Drive data_root directly")

In [ ]:
# ── 🔎 OCR PaddleOCR tiếng Việt — đọc lại MỌI keyframe (shard song song) ──
# Round-51: shard ĐÃ ĐẶT SẴN theo tên file (bản a/b/c) — không chỉnh gì cả,
# chỉ Run all. Mỗi phiên gánh 1/N số video;
# kết quả từng video đổ chung về MỘT kho Drive (ocr-v2-partial) mỗi 10 phút —
# phiên chết chỉ mất tối đa 10 phút công, chạy lại là tự nối tiếp.
# Phiên nào hoàn tất mà thấy KHO ĐỦ toàn bộ video sẽ tự FINALIZE (rebuild
# BM25 text-index + hoán đổi vào artifacts thật, có backup đường lui).
SHARD_INDEX = 2
SHARD_TOTAL = 3
import os, shutil, subprocess, sys, threading
import time as _tm
from pathlib import Path

def _run(*args):
    print("$", " ".join(map(str, args)), flush=True)
    _pp = subprocess.Popen([sys.executable, "-u", *map(str, args)],
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True)
    for _ln in _pp.stdout:
        print(_ln, end="", flush=True)
    if _pp.wait() != 0:
        raise RuntimeError(f"lệnh lỗi (exit {_pp.returncode})")

# Round-57 (audit): danh sách video phải lấy từ bản LOCAL đã materialize + tự
# vá (cell trước) — listing Drive FUSE có thể trả THIẾU/RỖNG (round-28), và
# _vids ngắn sẽ khiến prune xóa nhầm bài thật + finalize giao kho thiếu bài.
_mk_local = Path("/content/data/map-keyframes")
_mk = _mk_local if _mk_local.is_dir() else (PROJECT / "data" / "map-keyframes")
_vids = sorted(_q.stem for _q in _mk.glob("*.csv"))
assert len(_vids) > 800, (
    f"map-keyframes chỉ liệt kê {len(_vids)} video — listing bất thường, "
    "chạy lại ô này (tuyệt đối không finalize với danh sách thiếu).")
_my = _vids[SHARD_INDEX::SHARD_TOTAL]
print(f"Shard {SHARD_INDEX + 1}/{SHARD_TOTAL}: {len(_my)}/{len(_vids)} video")

_partial = PROJECT / "artifacts" / "ocr-v2-partial"
# Round-52 (live 05 run 24/08): N phiên cùng mkdir một lúc → Google Drive tạo
# NHIỀU thư mục TRÙNG TÊN (Drive cho phép trùng tên!), mỗi phiên đổ bài vào
# một bản → không phiên nào đếm đủ, FINALIZE không bao giờ nổ. Hai lớp chống:
# (1) chỉ ca 1 được TẠO kho — các ca sau ĐỢI thấy kho rồi mới vào;
if not (SHARD_INDEX == 0 or _partial.exists()):
    for _w in range(40):                     # tới 20 phút
        if _partial.exists():
            break
        print(f"⏳ đợi ca 1 tạo kho chung ({_w * 30}s) — cứ để yên ...", flush=True)
        _tm.sleep(30)
    else:
        print("⚠ 20 phút không thấy kho chung — đành tự tạo (hãy chắc ca 1 đang chạy).")
_partial.mkdir(parents=True, exist_ok=True)
# (2) lưới an toàn: gộp mọi kho sinh đôi "<tên> (1)"… về bản chính rồi xóa —
# nhờ vậy chạy lại ô này trên MỘT phiên là tự lành + finalize được.
for _dup in sorted(_partial.parent.glob(_partial.name + " (*")):
    if not _dup.is_dir():
        continue
    _n_dup = 0
    for _f in _dup.glob("*.json"):
        _d = _partial / _f.name
        if not _d.exists() or _d.stat().st_size < _f.stat().st_size:
            shutil.copy2(_f, _d)
            _n_dup += 1
    shutil.rmtree(_dup, ignore_errors=True)
    print(f"⚠ Gộp kho trùng tên '{_dup.name}': +{_n_dup} video", flush=True)
# Round-53 (live 05 run 24/08): CVP_PATHS__ARTIFACTS_ROOT của cell 4 vẫn trỏ
# VÀO DRIVE → "staging local" hóa ra là artifacts/captions THẬT trên Drive:
# store thật bị rmtree, N phiên đua nhau tạo staging sinh đôi trùng tên, và
# job còn ghi đè text_index trận đấu bằng bản nửa vời. Staging phải nằm
# trên ĐĨA LOCAL của VM — Drive chỉ nhận kết quả qua syncer + FINALIZE.
_la = Path("/content/artifacts")
_la.mkdir(parents=True, exist_ok=True)
os.environ["CVP_PATHS__ARTIFACTS_ROOT"] = str(_la)
# Round-55 (live 05a run 24/08): script aux ĐỌC catalog/manifest.parquet từ
# artifacts root — staging local rỗng phải kéo bản Drive về trước, không thì
# chết ngay "Catalog missing" (di chứng của round-53).
_drv_cat = PROJECT / "artifacts" / "catalog"
_ensure_drive()                          # round-57: mount phải sống trước FUSE I/O
assert (_drv_cat / "manifest.parquet").exists(), (
    "Thiếu artifacts/catalog/manifest.parquet trên Drive — chạy nb01 trước.")
for _try in (1, 2, 3):                   # round-57: copy đầu phiên cũng phải lì đòn
    try:
        shutil.copytree(_drv_cat, _la / "catalog", dirs_exist_ok=True)
        if (_la / "catalog" / "manifest.parquet").exists():
            print("catalog: staged về local")
            break
    except OSError as _e:
        print(f"   ⚠ copy catalog lỗi: {_e!r}")
    _ensure_drive()
    _tm.sleep(15)
else:
    raise RuntimeError("Không kéo được catalog về local sau 3 lần — chạy lại ô này.")
_job_local = _la / "ocr"
# Round-57 (audit): KHÔNG rmtree staging và KHÔNG đè file local bằng bản Drive
# — bài local (ghi nguyên tử) là bản đáng tin nhất; sync chốt lỗi rồi chạy lại
# ô này sẽ không mất bài nữa. Chỉ bù những file THIẾU từ kho chung.
_job_local.mkdir(parents=True, exist_ok=True)
for _try in (1, 2, 3):
    try:
        for _f in _partial.glob("*.json"):           # resume xuyên phiên
            _d = _job_local / _f.name
            if not _d.exists():
                shutil.copy2(_f, _d)
        break
    except OSError as _e:
        print(f"   ⚠ seed staging lỗi: {_e!r} — thử lại {_try}/3")
        _ensure_drive()
        _tm.sleep(15)
else:
    raise RuntimeError("Không seed được staging từ kho chung — chạy lại ô này.")
_vidset = set(_vids)

def _prune_staging():
    # Round-54: kho chung có thể lẫn rác sau các thao tác dọn tay trên Drive
    # web (bản trùng tên "xxx (1).json", file up nhầm chỗ…) — gạt khỏi staging
    # để store/text-index không nuốt phải video ma.
    # Round-57 (audit): file RÁCH (phiên chết giữa lúc sync) cũng phải bị gạt
    # — resume chỉ nhìn tên file, bản rách sẽ bị khóa vĩnh viễn vào kho trận
    # đấu nếu để lọt; xóa để job tính lại rồi sync đè bản lành lên kho chung.
    import json as _json
    for _p in list(_job_local.iterdir()):
        _ok = _p.is_file() and _p.suffix == ".json" and _p.stem in _vidset
        if _ok:
            try:
                with open(_p, encoding="utf-8") as _fh:
                    _json.load(_fh)
            except Exception:
                _ok = False
        if not _ok:
            shutil.rmtree(_p, ignore_errors=True) if _p.is_dir() else _p.unlink()
            print("   bỏ qua file lạ/rách trong kho:", _p.name, flush=True)

_prune_staging()
print(f"Resume: {len(list(_job_local.glob('*.json')))} video đã xong từ trước")

_stop_sync = False

def _syncer():
    # Round-57 (audit): daemon DriveFS có thể CHẾT giữa phiên dài (Errno 107)
    # — syncer câm lặng sẽ âm thầm ngừng đổ bài về Drive hàng chục giờ. Giờ nó
    # tự hồi sinh mount trước mỗi lượt và LA LỚN khi sync hỏng.
    while not _stop_sync:
        _tm.sleep(600)
        try:
            _ensure_drive()
            _n = 0
            for _f in _job_local.glob("*.json"):
                _d = _partial / _f.name
                if not _d.exists() or _d.stat().st_size != _f.stat().st_size:
                    shutil.copy2(_f, _d)
                    _n += 1
            _p_new = 0
            for _f in _partial.glob("*.json"):   # round-58: KÉO chiều về — học
                _d = _job_local / _f.name        # ngay bài các ca khác vừa xong
                if not _d.exists():              # để job tự skip, không trùng việc
                    shutil.copy2(_f, _d)
                    _p_new += 1
            if _n or _p_new:
                print(f"SYNC {_tm.strftime('%H:%M')}: đẩy {_n} / kéo {_p_new} video",
                      flush=True)
        except Exception as _e:  # noqa: BLE001 — không được giết job vì sync
            print(f"⚠ SYNC {_tm.strftime('%H:%M')} LỖI: {_e!r} — thử lại sau 10 phút",
                  flush=True)

# ── Cài PaddleOCR (PP-OCRv5) + smoke test 1 keyframe THẬT ──────────────
print("Cài PaddleOCR ...", flush=True)
_rc0 = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                       "paddlepaddle-gpu", "paddleocr"]).returncode
_gpu_ok = subprocess.run(
    [sys.executable, "-c",
     "import paddle; assert paddle.device.is_compiled_with_cuda()"],
    capture_output=True).returncode == 0
if _rc0 != 0 or not _gpu_ok:
    print("⚠ paddlepaddle-gpu không nhận CUDA — chuyển bản CPU (chậm hơn nhưng "
          "vẫn về đích).", flush=True)
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-q", "-y",
                    "paddlepaddle-gpu"])
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "paddlepaddle", "paddleocr"])
_kf_smoke = sorted(Path("/content/data/keyframes").glob("*/*.jpg"))
assert _kf_smoke, "Không thấy keyframe local — cell materialize chưa chạy?"
_smoke = subprocess.run(
    [sys.executable, "-c",
     "import os; os.environ['CVP_OCR__ENGINE'] = 'paddle'; "
     "from cvp.config import load_settings; "
     "from cvp.auxindex.ocr import _build_engine; "
     "eng = _build_engine(load_settings()); "
     f"print('SMOKE OCR:', repr(eng.read({str(_kf_smoke[0])!r})[:150]))"])
if _smoke.returncode != 0:
    raise RuntimeError("PaddleOCR smoke test THẤT BẠI — dừng TRƯỚC khi tốn giờ "
                       "GPU. Chụp log ô này gửi Claude.")
print("✅ PaddleOCR sẵn sàng.", flush=True)
threading.Thread(target=_syncer, daemon=True).start()
# Round-59: PaddleOCR đọc tiếng Việt chuẩn hơn hẳn EasyOCR (kho cũ) — kỳ vọng
# lật kênh OCR từ tín hiệu nhiễu (tuner từng dìm 0.35 → 0.01) thành vũ khí.
os.environ["CVP_OCR__ENGINE"] = "paddle"
_run(REPO_DIR / "scripts" / "03_build_aux_indexes.py", "--ocr",
     "--videos", *_my)
_stop_sync = True
for _try in (1, 2, 3):                   # đợt sync chốt của shard — phải lì đòn
    try:
        _ensure_drive()                  # round-57: mount có thể đã chết giữa job
        for _f in _job_local.glob("*.json"):
            _d = _partial / _f.name
            if not _d.exists() or _d.stat().st_size != _f.stat().st_size:
                shutil.copy2(_f, _d)
        break
    except OSError as _e:
        print(f"   ⚠ sync chốt lỗi: {_e!r} — thử lại {_try}/3")
        _tm.sleep(20)
else:
    raise RuntimeError("Sync chốt về Drive thất bại — KHÔNG xóa runtime! "
                       "Chạy lại ô này để đẩy nốt kết quả local lên Drive.")
_done = {_p.stem for _p in _partial.glob("*.json")}
_missing = [_v for _v in _vids if _v not in _done]
print(f"Shard xong. Kho chung: {len(_vids) - len(_missing)}/{len(_vids)} video")
if _missing:
    print(f"   còn thiếu (vd): {_missing[:5]}")

# ── FINALIZE: shard nào thấy kho đủ sẽ chốt hạ (có khóa chống chạy đôi) ──
# Round-54: đếm theo TÊN video (stem) — file trùng tên/lạ không thổi phồng số.
if not _missing:
    _lock = _partial / "_finalize.lock"
    _done_mark = _partial / "_finalize.done"
    # Round-56: khóa phải phân biệt "đã xong" / "đang chạy" / "chết giữa chừng"
    # — finalize crash không được để khóa mồ côi chặn vĩnh viễn.
    # Round-57 (audit): exists→write KHÔNG nguyên tử qua FUSE (cache trễ giữa
    # các máy) → khóa mang TOKEN riêng + xác nhận lại sau 90s + nhịp tim làm
    # mới khóa giữa các bước dài (finalize thật có thể >2h) + ngưỡng cũ 6h.
    import uuid as _uuid
    _token = _uuid.uuid4().hex

    def _touch_lock():
        _lock.write_text(_token, encoding="utf-8")

    _lock_fresh = False
    if _lock.exists():
        try:
            _lock_fresh = (_tm.time() - _lock.stat().st_mtime) < 6 * 3600
        except OSError:
            pass
    if _done_mark.exists():
        print("Finalize đã hoàn tất từ trước — không cần làm lại.")
    elif _lock.exists() and _lock_fresh:
        print("Finalize đang được phiên khác chạy (khóa <6h) — bỏ qua. Nếu chắc "
              "chắn không phiên nào khác đang chạy: xóa _finalize.lock trong "
              "kho chung trên Drive rồi chạy lại ô này.")
    else:
        if _lock.exists():
            print("Khóa finalize cũ (>6h) mà chưa có dấu hoàn tất — tiếp quản chốt lại.")
        _touch_lock()
        print("FINALIZE: giành khóa — chờ 90s xác nhận không phiên nào giành cùng lúc ...")
        try:
            _tm.sleep(90)
            _mine = False
            try:
                _mine = _lock.read_text(encoding="utf-8").strip() == _token
            except OSError:
                pass
            if not _mine:
                raise RuntimeError(
                    "Phiên khác giành khóa finalize cùng lúc — phiên này NHƯỜNG "
                    "(KHÔNG phải lỗi; theo dõi phiên kia là được).")
            print("FINALIZE: rebuild BM25 + hoán đổi artifacts ...")
            # Round-56: staging phải ĐẦY ĐỦ THẬT — copytree qua FUSE từng giao
            # thiếu (863/873 live). Kéo lại vài lượt rồi tự caption bù phần thiếu.
            _ensure_drive()
            _still = []
            for _try in (1, 2, 3):
                for _f in _partial.glob("*.json"):   # chỉ bù file THIẾU — không
                    _d = _job_local / _f.name        # đè bản local lành bằng
                    if not _d.exists():              # bản Drive có thể rách
                        shutil.copy2(_f, _d)
                _prune_staging()
                _still = [_v for _v in _vids
                          if not (_job_local / (_v + ".json")).is_file()]
                if not _still:
                    break
                print(f"   staging thiếu {len(_still)} video (vd {_still[:3]}) — kéo lại {_try}/3 ...")
                _tm.sleep(30)
            if _still:
                print(f"   tự chạy bù {len(_still)} video thiếu ...")
                _run(REPO_DIR / "scripts" / "03_build_aux_indexes.py",
                     "--ocr",
                     "--videos", *_still)
                for _v in _still:                    # trả bản bù về kho chung
                    _f = _job_local / (_v + ".json")
                    if _f.is_file():
                        shutil.copy2(_f, _partial / _f.name)
            # Round-56: kho aux nào kéo thiếu → index trận đấu âm thầm yếu đi.
            for _aux in ("ocr", "asr", "captions"):  # BM25 cần đủ các kho aux
                _src = PROJECT / "artifacts" / _aux
                if _aux == "ocr" or not _src.is_dir():
                    continue
                for _try in (1, 2, 3):
                    shutil.copytree(_src, _la / _aux, dirs_exist_ok=True)
                    # round-57: listing Drive có thể TRẢ THIẾU y hệt copy —
                    # neo số cần vào danh sách video, không tin listing suông.
                    _need = max(len(list(_src.glob("*.json"))), len(_vids))
                    _got = len(list((_la / _aux).glob("*.json")))
                    if _got >= _need:
                        print(f"   staging {_aux}: {_got}/{_need} file")
                        break
                    print(f"   staging {_aux} thiếu ({_got}/{_need}) — thử lại {_try}/3 ...")
                    _tm.sleep(60)
                else:
                    raise RuntimeError(
                        f"Kéo kho {_aux} về máy mãi vẫn thiếu — Drive trục trặc; "
                        "chạy lại ô này sau ít phút (khóa sẽ tự nhả).")
            _touch_lock()                # nhịp tim khóa trước bước rebuild dài
            _run(REPO_DIR / "scripts" / "03_build_aux_indexes.py",
                 "--text-index", "--force-text-index")
            _ensure_drive()              # round-57: mount phải sống trước hoán đổi
            _touch_lock()
            _drv_job = PROJECT / "artifacts" / "ocr"
            _bak = PROJECT / "artifacts" / "ocr-easyocr-backup"
            if _drv_job.exists() and not _bak.exists():
                _drv_job.rename(_bak)
                print("Đã cất bản cũ →", _bak)
            for _d in ("ocr", "text_index"):
                _dst = PROJECT / "artifacts" / _d
                if _dst.exists():
                    shutil.rmtree(_dst)
                shutil.copytree(_la / _d, _dst)
                # Round-56: upload cũng đếm lại — copy bù nếu Drive nuốt thiếu
                _n_src = sum(1 for _q in (_la / _d).rglob("*") if _q.is_file())
                _n_dst = sum(1 for _q in _dst.rglob("*") if _q.is_file())
                if _n_dst < _n_src:
                    shutil.copytree(_la / _d, _dst, dirs_exist_ok=True)
                    _n_dst = sum(1 for _q in _dst.rglob("*") if _q.is_file())
                print(f"   → Drive: {_dst} ({_n_dst}/{_n_src} file)")
            _done_mark.write_text(_tm.strftime("%Y-%m-%d %H:%M"), encoding="utf-8")
        except BaseException:
            # round-57: chỉ nhả khóa nếu vẫn là khóa CỦA MÌNH — thua cuộc đua
            # thì tuyệt đối không được gỡ khóa của phiên thắng.
            try:
                if _lock.read_text(encoding="utf-8").strip() == _token:
                    _lock.unlink(missing_ok=True)   # nhả khóa để chạy lại được ngay
            except OSError:
                pass
            raise
        print("XONG TOÀN BỘ — artifacts đã nâng cấp. Đo lại bằng nb04 (L2).")
else:
    print("Kho chưa đủ — chờ các shard khác (hoặc chạy lại phiên để nối tiếp).")

In [ ]:
# ── L6 · 🫀 Giữ phiên sống sau khi Lab xong (bấm ⏹ của ô này để dừng) ──
# Round-46: phiên Lab từng bị Colab thu hồi vì "không hoạt động". Kernel bận
# chạy ô này = hoạt động. Kết quả các stage đã được LƯU THẲNG LÊN DRIVE ngay
# khi có (bench_full.json / best_weights.json / lane_ab.json / lab_full/),
# nên dù phiên chết cũng không mất bài — ô này chỉ giữ máy ảo cho bạn quay
# lại chạy thêm stage. GIỮ TAB TRÌNH DUYỆT MỞ trong lúc Lab chạy.
import time as _tm
print("🫀 Lab watchkeeper — phiên được giữ sống.")
try:
    _n = 0
    while True:
        _tm.sleep(30)
        _n += 1
        if _n % 10 == 0:
            print(f"🫀 {_tm.strftime('%H:%M')} phiên sống")
except KeyboardInterrupt:
    print("⏹ Dừng — phiên sẽ tính là nhàn rỗi từ giờ.")